# 03: Any substrate, any organism, any product

Notebooks 01 and 02 explain the ideas with E. coli, glucose and ethanol. This one shows the generic tool: **feed substrate X to organism Y and ask for product Z**.

* **Organism** = a model. Pass a BiGG id (`iML1515` E. coli, `iMM904` yeast, `iYS1720` Salmonella, ... see [bigg.ucsd.edu/models](http://bigg.ucsd.edu/models)), or a path to any SBML/JSON/MAT model file.
* **Substrate / product** = a name (`glucose`, `Ethanol`), a metabolite id (`etoh`), or an exchange reaction id (`EX_etoh_e`).

The same thing works from the terminal:

```bash
cellfactory run --model iYS1720 --substrate glycerol --product ethanol --anaerobic --knockouts 2 --checks
cellfactory find --model iML1515 lycopene   # what is this compound called in the model?
```

In [1]:
import warnings
import pandas as pd
from cellfactory import analyze, find_metabolites, load_model

warnings.filterwarnings("ignore", message="Solver status is 'infeasible'")

## One scenario

`analyze` sets up the medium (swaps the model's default sugar for your substrate), finds the product's exchange reaction, and reports yields. Yields are in mol product per mol substrate, plus the share of the substrate's carbon atoms that ends up in the product, which lets you compare a 5-carbon sugar with a 6-carbon one fairly.

In [2]:
report = analyze("iJO1366", substrate="xylose", product="ethanol", aerobic=False)
print(report.to_text())

iJO1366: EX_xyl__D_e -> EX_etoh_e (anaerobic, uptake 10.0 mmol/gDW/h)
  growth rate                 0.168 /h
  theoretical max yield       1.667 mol/mol
  guaranteed yield @ growth   0.697 mol/mol  (28% of carbon)
  best-case yield @ growth    0.703 mol/mol


## Compare scenarios

Xylose is the main sugar in plant waste (straw, wood), so making fuel from it rather than food-grade glucose is a big deal. Here's a small grid: two organisms, three substrates, one product. `None` means the organism can't grow on that substrate (e.g. it lacks a transporter or pathway for it).

In [3]:
rows = []
for organism in ["iJO1366", "iYS1720"]:
    model = load_model(organism)
    for substrate in ["glucose", "xylose", "glycerol"]:
        for aerobic in [True, False]:
            try:
                r = analyze(model.copy(), substrate, "ethanol", aerobic=aerobic)
                p = r.production
                rows.append((organism, substrate, aerobic, p.growth, p.max_theoretical_yield, p.guaranteed_yield, r.carbon_yield))
            except (KeyError, ValueError) as err:
                rows.append((organism, substrate, aerobic, None, None, None, None))
pd.DataFrame(rows, columns=["model", "substrate", "aerobic", "growth", "max_yield", "guaranteed_yield", "carbon_yield"]).round(3)

,model,substrate,aerobic,growth,max_yield,guaranteed_yield,carbon_yield
0,iJO1366,glucose,True,0.982,2.000,0.000,0.000
1,iJO1366,glucose,False,0.242,2.000,0.804,0.268
2,iJO1366,xylose,True,0.806,1.667,0.000,0.000
3,iJO1366,xylose,False,0.168,1.667,0.697,0.279
4,iJO1366,glycerol,True,0.563,1.167,0.000,0.000
5,iJO1366,glycerol,False,0.078,1.167,0.488,0.325
6,iYS1720,glucose,True,1.014,2.000,0.000,0.000
7,iYS1720,glucose,False,0.202,2.000,0.851,0.284
8,iYS1720,xylose,True,0.828,1.667,0.000,0.000
9,iYS1720,xylose,False,0.127,1.667,0.739,0.296


Notice the pattern: aerobic growth is fast but makes no ethanol (respiration burns everything to CO₂); anaerobic growth is slow but forced to make fermentation products.

## Finding a compound's id

Every model names things a little differently. `find_metabolites` searches ids and names.

In [4]:
core = load_model("e_coli_core")
[(m.id, m.name, m.formula) for m in find_metabolites(core, "succin")]

[('succ_c', 'Succinate', 'C4H4O4'),
 ('succ_e', 'Succinate', 'C4H4O4'),
 ('succoa_c', 'Succinyl-CoA', 'C25H35N7O19P3S')]

## What if the organism can't make the product?

*E. coli* has no lycopene pathway, so the tool says so instead of guessing. To engineer it in, you'd add the missing reactions (enzymes borrowed from another organism) with `model.add_reactions`, then run `analyze` again: the product now has a metabolite, and `analyze` adds a demand reaction to drain it.

In [5]:
try:
    analyze("e_coli_core", "glucose", "lycopene")
except KeyError as err:
    print(err.args[0])

'lycopene' is not a metabolite in e_coli_core. Nothing similar is in the model. The organism may lack the pathway; add the reactions with model.add_reactions() first.


## Knockouts and sanity checks, for any scenario

`max_knockouts` runs the greedy search from notebook 01, `run_checks=True` runs the checks from notebook 02. The E. coli lab comparison only appears for E. coli models; for other organisms you get the list of predicted essential genes to read through.

In [6]:
report = analyze("e_coli_core", "glucose", "succinate", aerobic=False, max_knockouts=2, run_checks=True)
print(report.to_text())

e_coli_core: EX_glc__D_e -> EX_succ_e (anaerobic, uptake 10.0 mmol/gDW/h)
  growth rate                 0.212 /h
  theoretical max yield       1.391 mol/mol
  guaranteed yield @ growth   0.000 mol/mol  (0% of carbon)
  best-case yield @ growth    0.002 mol/mol

Greedy knockout search:
knocked_out                                                    genes   growth  guaranteed_yield
         []                                                       [] 0.211663          0.000000
      [PFL]               [b0902, b0903, b2579, b3114, b3951, b3952] 0.179985          0.067747
 [PFL, PYK] [b0902, b0903, b1676, b1854, b2579, b3114, b3951, b3952] 0.103473          0.738544

Free-energy checks: 3 rows
                       check  value   ok
growth with no uptake at all    0.0 True
       ATP made from nothing    0.0 True
growth with no carbon source    0.0 True

Mass-imbalanced reactions: 0 rows
  (none)

Gene essentiality vs E. coli lab data (aerobic glucose): 18 rows
 gene                       